# K-12 Spending: Salaries, Pensions, Benefits and Supplies
Where does a school dollar go? The Department of Education collects every district's, county office's and charter school's year-end accounts under one code structure (SACS), with each dollar coded by **what was bought** (the object: teacher pay, pensions, supplies...) and **what it was for** (the function: classroom, administration, buses...). This page uses the statewide totals for school years 2003-04 to the latest (`scripts/fetch_cde_sacs.py`). Fiscal year is the year the school year ends.

**Read this first**
- **Scope:** district, county-office and charter spending that files in SACS (almost all of it), all funds except pass-through, internal-service and fiduciary funds. Transfers between agencies and funds, and indirect-cost transfers, are removed so money is not counted twice. These are **unaudited actuals**. Community colleges are not included. Charter schools that file only the short alternative form are not in the totals.
- **Pensions** here are what school *employers* pay into the teachers' (STRS) and classified employees' (PERS) retirement systems. California's budget also pays into the teachers' retirement fund on the state's own account; that is not in school ledgers and is shown separately below.
- **Matches Census, roughly:** after removing debt principal (which Census leaves out), the totals here are within 1 to 5% of the Census school-spending figures on the school page (the gap widens after FY2019, when Census dropped about 900 charter schools that SACS still includes), but they are different surveys, so use this page for the *breakdown* and that one for the level.
- Groupings (for example which object codes count as pensions) are in `src/ca_taxes/school.py`.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, line_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
sp = pd.read_csv(PROCESSED / "sacs_spending.csv")
fn = pd.read_csv(PROCESSED / "sacs_function_group.csv")
SACS = "Source: California Department of Education, SACS unaudited actuals (statewide totals)"
from ca_taxes.school import GROUPS, FUNCTION_ORDER
d = add_views(sp.groupby(["fy", "group"], as_index=False).amount_k.sum())
stacked_area_toggle(d, "group", "K-12 spending by object: where the money goes", SACS + ". All funds except pass-through; transfers removed.", order=GROUPS).show()

## For every dollar of salary
Benefits paid per dollar of pay (certificated plus classified salaries), by type. Pension contributions rose sharply with the state's 2014 plan to close the teachers' retirement funding gap, which raised the rate employers pay.

In [2]:
t = sp.pivot_table(index="fy", columns="group", values="amount_k", aggfunc="sum")
sal = t["Teachers' salaries"] + t["Other certificated salaries"] + t["Classified staff salaries"]
r = pd.DataFrame({"Pensions": t["Pensions"] / sal, "Health insurance": t["Health insurance"] / sal, "Other benefits": t["Other benefits"] / sal}) * 100
fig = go.Figure()
for i, c in enumerate(r.columns):
    fig.add_trace(go.Scatter(x=r.index, y=r[c], name=c, mode="lines+markers", stackgroup="one", line=dict(color=PALETTE[[2, 0, 5][i]], width=0.5), hovertemplate="%{y:.1f} cents"))
fig.update_layout(yaxis_title="Cents of benefits per dollar of salary", xaxis_title="Fiscal year (ending June 30)")
base_layout(fig, "Benefit cost per dollar of salary", "Employer-paid pensions (STRS and PERS), health insurance and other benefits over total salaries. " + SACS)
fig.show()

**What the data show:** employer pension contributions (STRS and PERS) were about 8.5 cents per dollar of salary from 2004 to 2014 and about 25 cents by 2024, rising each year from 2015 to 2019, after the state's 2014 plan to close the teachers' retirement funding gap raised employer rates. As a share of all K-12 spending, pensions rose from 4.4% to 11.0%. Teachers' salaries fell from 31% to 23% of spending (and from 64% to 50% of classroom-instruction spending) while health insurance stayed roughly flat at 12 to 15 cents per dollar of pay. Spending on books and supplies stayed near 4% of the total (5% of classroom instruction). These shares are of total spending including capital projects and debt service; the 2020-2024 totals also include one-time federal pandemic relief.

## Staff pay, benefits and everything else, in detail
The next three charts split the groups into their parts: who is paid, which benefits, and what the non-staff dollars buy.

In [3]:
def detail(groups, title, sub):
    x = sp[sp.group.isin(groups)].groupby(["fy", "category"], as_index=False).amount_k.sum()
    stacked_area_toggle(add_views(x), "category", title, sub).show()
detail(["Teachers' salaries", "Other certificated salaries", "Classified staff salaries"], "Staff pay by job type", SACS)
detail(["Pensions", "Health insurance", "Other benefits"], "Employee benefits by type", SACS + ". STRS and PERS are employer contributions only.")
detail(["Books and supplies", "Food", "Services and operating", "Capital outlay", "Debt service and other"], "Non-staff spending by type", SACS)

## What the money is for
The same spending by function. Instruction is the classroom (including special education); the other functions are the supports around it.

In [4]:
f = fn.groupby(["fy", "function_group"], as_index=False).amount_k.sum()
stacked_area_toggle(add_views(f), "function_group", "K-12 spending by function", SACS, order=[g for g in FUNCTION_ORDER if g in set(f.function_group)]).show()

## Inside the classroom
What classroom instruction itself is made of: of every dollar coded to instruction (including special education), how much is teacher pay, other staff pay, pensions, other benefits, and supplies?

In [5]:
ins = fn[fn.function_group == "Classroom instruction (incl. special ed)"].groupby(["fy", "group"], as_index=False).amount_k.sum()
stacked_area_toggle(add_views(ins), "group", "What classroom instruction is spent on", SACS + ". Function 1000-1999.", order=[g for g in GROUPS if g in set(ins.group)]).show()

## A school dollar, then and now
Share of total K-12 spending in each category, in the first and latest years and one in between.

In [6]:
tot = sp.groupby(["fy", "group"]).amount_k.sum().unstack()
share = tot.div(tot.sum(axis=1), axis=0) * 100
yrs = [share.index.min(), 2014, share.index.max()]
fig = go.Figure()
for i, g in enumerate(GROUPS):
    fig.add_trace(go.Bar(x=[f"FY{y}" for y in yrs], y=[share.loc[y, g] if g in share else 0 for y in yrs], name=g, marker_color=PALETTE[i % len(PALETTE)],
                         hovertemplate="%{y:.1f}%"))
fig.update_layout(barmode="stack", yaxis_title="% of total K-12 spending", yaxis_ticksuffix="%")
base_layout(fig, "Share of K-12 spending by object", "Share of the cleaned statewide total. " + SACS, height=620)
fig.show()

## The state's own pension contribution
The state budget pays into the teachers' retirement fund separately from what school employers contribute. The two together are the public cost of school-employee pensions through these systems (STRS and PERS employer contributions plus the state's STRS payment). The state's payment covers STRS members in community colleges as well as K-12, so it slightly overstates the K-12 share.

In [7]:
e = pd.read_csv(PROCESSED / "state_expenditures.csv")
st = e[e.department.isin(["Teachers Retirement System Contributions", "Contributions to Teacher's Retirement Fund"])].groupby("fy", as_index=False).amount_k.sum()
st = st[st.fy <= LAST_ACTUAL_FY]
schools = sp[sp.group == "Pensions"].groupby("fy", as_index=False).amount_k.sum()
both = pd.concat([schools.assign(payer="School employers (STRS and PERS)"), st[st.fy.isin(schools.fy)].assign(payer="State budget (STRS)")])
stacked_area_toggle(add_views(both), "payer", "Pension money for school employees: school employers and the state",
                    "School ledgers: " + SACS + ". State: Legislative Analyst's Office, contributions to the teachers' retirement fund.",
                    order=["School employers (STRS and PERS)", "State budget (STRS)"]).show()
st_all = add_views(st)
fig = go.Figure(go.Scatter(x=st_all.fy, y=st_all.real_b, mode="lines+markers", line=dict(color=PALETTE[2]), marker=dict(size=4), hovertemplate="$%{y:.1f}B"))
fig.update_layout(yaxis_title=f"$B, {base_fy()} dollars", xaxis_title="Fiscal year (ending June 30)", showlegend=False)
base_layout(fig, "State contributions to teachers' retirement, since 1985", "Real dollars. FY2019 is the series' peak; the reason is not examined here. Source: LAO / DOF", height=460)
fig.show()